In [ ]:
# Day 18 - Using Fine-tuned LoRA Model
!pip install -q peft accelerate bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

In [ ]:
# 1. Load Base Model + LoRA Adapter
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "ethiopian_lora_adapter"   # from Day 17

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True
)

# Load LoRA Adapter
model = PeftModel.from_pretrained(base_model, adapter_path)
print("✅ Fine-tuned model loaded successfully!")

In [ ]:
# 2. Generation Function
def generate_response(prompt, max_new_tokens=300, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Test
test_prompt = """You are a helpful Ethiopian AI assistant.

User: What is the capital of Ethiopia and why is it important?
Assistant:"""

print(generate_response(test_prompt))

In [ ]:
# 3. Improved Ethiopian Assistant
def ethiopian_assistant(question):
    prompt = f"""You are a knowledgeable and friendly Ethiopian AI assistant.

Question: {question}

Answer:"""
    
    return generate_response(prompt, max_new_tokens=250, temperature=0.65)

# Test
print(ethiopian_assistant("Tell me about Ethiopian culture and history."))
print(ethiopian_assistant("What should I know before visiting Ethiopia?"))